<a href="https://colab.research.google.com/github/Luigi2077-spec/AplicacionWeb_ES04/blob/main/AGENCIA_VIAJES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema de Gestión para Agencia de Viajes
**Autor:** Luis
**Descripción:** Aplicación de consola desarrollada en Python para la gestión de reservas y destinos turísticos.

### Características Técnicas
* **Base de Datos:** Integración y consumo de Oracle DB utilizando la librería `oracledb`.
* **Seguridad y Credenciales:** Encriptación de contraseñas (hashing) con `bcrypt`. Gestión segura de credenciales de base de datos inyectándolas en tiempo de ejecución mediante `google.colab.userdata`.
* **Arquitectura:** Estructuración del código basada en el patrón de diseño DAO (Data Access Object) para aislar la lógica de negocio del acceso a datos.
* **Control de Flujo:** Prevención de inyección SQL mediante el uso de variables *bind* parametrizadas y control estricto de transacciones (`commit`/`rollback`).

In [4]:
!pip install oracledb
!pip install bcrypt
import oracledb
import bcrypt
import getpass
from datetime import datetime
from google.colab import userdata # Importar para usar secretos

class Database:
    _instance = None

    def __init__(self):
        # Cargar credenciales desde los secretos de Colab
        self.server_ip = userdata.get('ORACLE_SERVER_IP')
        self.user = userdata.get('ORACLE_USER')
        self.password = userdata.get('ORACLE_PASSWORD')
        self.dsn = oracledb.makedsn(self.server_ip, 1521, service_name="XEPDB1")

    def connect(self):
        try:
            return oracledb.connect(user=self.user, password=self.password, dsn=self.dsn)
        except Exception as e:
            print(f"Error conectando a Oracle: {e}")
            return None

    @classmethod
    def get_instance(cls):
        if cls._instance is None:
            cls._instance = Database()
        return cls._instance

def hash_password(password):
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()

def verify_password(password, stored_hash):
    try:
        return bcrypt.checkpw(password.encode(), stored_hash.encode())
    except Exception as e:
        print(f"Error verifying password: {e}")
        return False

class UsuarioDAO:
    def __init__(self):
        self.db = Database.get_instance()

    def crear(self, rut, password, rol_codigo='CUS'):
        conn = self.db.connect()
        if not conn: return False
        cursor = conn.cursor()
        try:
            hashed = hash_password(password)
            sql = """
            INSERT INTO USUARIO (RUT_USUARIO, PASSWORD_HASH, CODIGO_ROL, ESTADO, FECHA_INGRESO)
            VALUES (:1, :2, :3, 'A', SYSDATE)
            """
            cursor.execute(sql, [rut, hashed, rol_codigo])
            conn.commit()
            return True
        except Exception as e:
            print(f"Error crear usuario: {e}")
            return False
        finally:
            cursor.close(); conn.close()

    def login(self, rut, password):
        conn = self.db.connect()
        if not conn: return None
        cursor = conn.cursor()
        try:
            sql = "SELECT RUT_USUARIO, PASSWORD_HASH, CODIGO_ROL FROM USUARIO WHERE RUT_USUARIO = :1 AND ESTADO = 'A'"
            cursor.execute(sql, [rut])
            row = cursor.fetchone()

            if row:
                rut_bd, pass_bd, rol_bd = row
                if verify_password(password, pass_bd):
                    rol_str = "admin" if str(rol_bd).strip() == "ADMIN" else "cliente"
                    return {"rut": rut_bd, "rol": rol_str}
            return None
        except Exception as e:
            print(f"Error login: {e}")
            return None
        finally:
            cursor.close(); conn.close()

class DestinoDAO:
    def __init__(self):
        self.db = Database.get_instance()

    def crear(self, nombre, descripcion, actividades, costo, f_ini, f_fin):
        conn = self.db.connect()
        if not conn: return False
        cursor = conn.cursor()
        try:
            sql = """
            INSERT INTO DESTINOS (NOMBRE, DESCRIPCION, ACTIVIDADES, COSTO, FECHA_INICIO, FECHA_FIN)
            VALUES (:1, :2, :3, :4, TO_DATE(:5, 'DD-MM-YYYY'), TO_DATE(:6, 'DD-MM-YYYY'))
            """
            cursor.execute(sql, [nombre, descripcion, actividades, costo, f_ini, f_fin])
            conn.commit()
            return True
        except Exception as e:
            print(f"Error crear destino: {e}")
            return False
        finally:
            cursor.close(); conn.close()

    def actualizar(self, id_dest, nombre, descripcion, actividades, costo, f_ini, f_fin):
        conn = self.db.connect()
        if not conn: return False
        cursor = conn.cursor()
        try:
            sql = """
            UPDATE DESTINOS SET
                NOMBRE=:1, DESCRIPCION=:2, ACTIVIDADES=:3, COSTO=:4,
                FECHA_INICIO=TO_DATE(:5, 'DD-MM-YYYY'), FECHA_FIN=TO_DATE(:6, 'DD-MM-YYYY')
            WHERE ID=:7
            """
            cursor.execute(sql, [nombre, descripcion, actividades, costo, f_ini, f_fin, id_dest])
            conn.commit()
            return True
        except Exception as e:
            print(f"Error actualizar destino: {e}")
            return False
        finally:
            cursor.close(); conn.close()

    def eliminar(self, id_dest):
        conn = self.db.connect()
        if not conn: return False
        cursor = conn.cursor()
        try:
            sql = "DELETE FROM DESTINOS WHERE ID=:1"
            cursor.execute(sql, [id_dest])
            conn.commit()
            return True
        except Exception as e:
            print(f"Error eliminar destino: {e}")
            return False
        finally:
            cursor.close(); conn.close()

    def listar(self):
        conn = self.db.connect()
        if not conn: return []
        cursor = conn.cursor()
        try:
            sql = "SELECT ID, NOMBRE, DESCRIPCION, ACTIVIDADES, COSTO, TO_CHAR(FECHA_INICIO, 'DD-MM-YYYY'), TO_CHAR(FECHA_FIN, 'DD-MM-YYYY') FROM DESTINOS ORDER BY ID"
            cursor.execute(sql)
            rows = cursor.fetchall()
            return [{"id": r[0], "nombre": r[1], "descripcion": r[2], "actividades": r[3], "costo": r[4], "f_ini": r[5], "f_fin": r[6]} for r in rows]
        finally:
            cursor.close(); conn.close()

    def buscar_por_fechas(self, f_ini_busqueda, f_fin_busqueda):
        conn = self.db.connect()
        if not conn: return []
        cursor = conn.cursor()
        try:
            sql = """
            SELECT ID, NOMBRE, DESCRIPCION, ACTIVIDADES, COSTO, TO_CHAR(FECHA_INICIO, 'DD-MM-YYYY'), TO_CHAR(FECHA_FIN, 'DD-MM-YYYY')
            FROM DESTINOS
            WHERE FECHA_INICIO >= TO_DATE(:1, 'DD-MM-YYYY')
              AND FECHA_FIN <= TO_DATE(:2, 'DD-MM-YYYY')
            ORDER BY FECHA_INICIO
            """
            cursor.execute(sql, [f_ini_busqueda, f_fin_busqueda])
            rows = cursor.fetchall()
            return [{"id": r[0], "nombre": r[1], "descripcion": r[2], "actividades": r[3], "costo": r[4], "f_ini": r[5], "f_fin": r[6]} for r in rows]
        finally:
            cursor.close(); conn.close()

class PaqueteDAO:
    def __init__(self):
        self.db = Database.get_instance()

    def crear_y_retornar_id(self, nombre_completo, fi, ff, precio_total):
        conn = self.db.connect()
        if not conn: return None
        cursor = conn.cursor()
        try:
            sql = """
            INSERT INTO PAQUETES (NOMBRE, FECHA_INICIO, FECHA_FIN, PRECIO_TOTAL)
            VALUES (:1, TO_DATE(:2, 'DD-MM-YYYY'), TO_DATE(:3, 'DD-MM-YYYY'), :4)
            """
            cursor.execute(sql, [nombre_completo, fi, ff, precio_total])

            cursor.execute("SELECT MAX(ID) FROM PAQUETES")
            row = cursor.fetchone()
            paquete_id = row[0] if row else None

            conn.commit()
            return paquete_id
        except Exception as e:
            print(f"Error crear paquete: {e}")
            return None
        finally:
            cursor.close(); conn.close()

    def listar(self):
        conn = self.db.connect()
        if not conn: return []
        cursor = conn.cursor()
        try:
            sql = "SELECT ID, NOMBRE, TO_CHAR(FECHA_INICIO, 'DD-MM-YYYY'), TO_CHAR(FECHA_FIN, 'DD-MM-YYYY'), PRECIO_TOTAL FROM PAQUETES ORDER BY ID"
            cursor.execute(sql)
            rows = cursor.fetchall()
            return [{"id": r[0], "nombre": r[1], "fecha_inicio": r[2], "fecha_fin": r[3], "precio_total": r[4]} for r in rows]
        finally:
            cursor.close(); conn.close()

class ReservaDAO:
    def __init__(self):
        self.db = Database.get_instance()

    def crear(self, rut_usuario, paquete_id):
        conn = self.db.connect()
        if not conn: return False
        cursor = conn.cursor()
        try:
            sql = "INSERT INTO RESERVAS (RUT_USUARIO, PAQUETE_ID, FECHA_RESERVA) VALUES (:1, :2, SYSDATE)"
            cursor.execute(sql, [rut_usuario, paquete_id])
            conn.commit()
            return True
        except Exception as e:
            print(f"Error crear reserva: {e}")
            return False
        finally:
            cursor.close(); conn.close()

    def listar_por_usuario(self, rut_usuario):
        conn = self.db.connect()
        if not conn: return []
        cursor = conn.cursor()
        try:
            sql = """
            SELECT r.ID, p.NOMBRE, TO_CHAR(r.FECHA_RESERVA, 'DD-MM-YYYY HH24:MI'), p.PRECIO_TOTAL
            FROM RESERVAS r
            JOIN PAQUETES p ON r.PAQUETE_ID = p.ID
            WHERE r.RUT_USUARIO = :1
            ORDER BY r.ID DESC
            """
            cursor.execute(sql, [rut_usuario])
            rows = cursor.fetchall()
            return [{"id": r[0], "paquete": r[1], "fecha": r[2], "precio": r[3]} for r in rows]
        finally:
            cursor.close(); conn.close()

class AgenciaService:
    def __init__(self):
        self.usuario_dao = UsuarioDAO()
        self.destino_dao = DestinoDAO()
        self.paquete_dao = PaqueteDAO()
        self.reserva_dao = ReservaDAO()

    def menu_admin_crud_destinos(self):
        while True:
            print("\n--- GESTION DE DESTINOS (ADMIN) ---")
            print("1. Crear Nuevo Destino")
            print("2. Modificar Destino Existente")
            print("3. Eliminar Destino")
            print("4. Listar Todos")
            print("0. Volver")
            op = input("Seleccione: ")

            if op == "0": break

            elif op == "1":
                print("\n[Creando Destino]")
                nom = input("Nombre: ")
                desc = input("Descripcion: ")
                act = input("Actividades: ")
                try:
                    costo = float(input("Costo: "))
                    fi = input("Fecha Inicio (DD-MM-YYYY): ")
                    ff = input("Fecha Fin (DD-MM-YYYY): ")
                    if self.destino_dao.crear(nom, desc, act, costo, fi, ff):
                        print("Destino creado.")
                except ValueError: print("Dato invalido.")

            elif op == "2":
                print("\n[Modificando Destino]")
                for d in self.destino_dao.listar():
                    print(f"ID: {d['id']} | {d['nombre']} ({d['f_ini']} a {d['f_fin']})")

                try:
                    did = int(input("Ingrese ID a modificar: "))
                    print("--- Ingrese nuevos datos ---")
                    nom = input("Nombre: ")
                    desc = input("Descripcion: ")
                    act = input("Actividades: ")
                    costo = float(input("Costo: "))
                    fi = input("Fecha Inicio (DD-MM-YYYY): ")
                    ff = input("Fecha Fin (DD-MM-YYYY): ")

                    if self.destino_dao.actualizar(did, nom, desc, act, costo, fi, ff):
                        print("Destino actualizado.")
                    else:
                        print("Error al actualizar.")
                except ValueError: print("Error de formato.")

            elif op == "3":
                print("\n[Eliminando Destino]")
                try:
                    did = int(input("Ingrese ID a eliminar: "))
                    if self.destino_dao.eliminar(did):
                        print("Destino eliminado.")
                    else:
                        print("No se pudo eliminar.")
                except: print("ID invalido.")

            elif op == "4":
                print("\n--- LISTA COMPLETA ---")
                for d in self.destino_dao.listar():
                    print(f"ID: {d['id']} | {d['nombre']} | ${d['costo']} | Fechas: {d['f_ini']} - {d['f_fin']}")

    def menu_cliente_armar_viaje(self, rut_cliente):
        print("\n--- ARMAR MI PROPIO VIAJE ---")
        print("Ingrese el rango de fechas en las que desea viajar.")
        f_ini_req = input("Fecha Inicio Deseada (DD-MM-YYYY): ")
        f_fin_req = input("Fecha Fin Deseada (DD-MM-YYYY): ")

        destinos_disponibles = self.destino_dao.buscar_por_fechas(f_ini_req, f_fin_req)

        if not destinos_disponibles:
            print("No hay destinos disponibles que encajen exactamente en esas fechas.")
            return

        print(f"\nSe encontraron {len(destinos_disponibles)} destinos en ese rango:")
        destinos_seleccionados = []
        costo_total = 0

        while True:
            print("\n--- Destinos Compatibles ---")
            for d in destinos_disponibles:
                print(f"ID: {d['id']} | {d['nombre']} | ${d['costo']} | ({d['f_ini']} al {d['f_fin']})")

            try:
                op_id = input("\nIngrese ID del destino a agregar (o ENTER para terminar): ")
                if op_id == "": break

                op_id = int(op_id)
                destino = next((x for x in destinos_disponibles if x['id'] == op_id), None)

                if destino:
                    destinos_seleccionados.append(destino['nombre'])
                    costo_total += destino['costo']
                    print(f"Agregado: {destino['nombre']}")
                else:
                    print("ID no valido o no corresponde a las fechas.")
            except ValueError:
                print("Debe ingresar un numero.")

        if not destinos_seleccionados:
            print("Cancelado.")
            return

        nombres_str = " + ".join(destinos_seleccionados)
        nombre_final = f"Pack Personalizado ({nombres_str})"

        print(f"\n--- Resumen del Viaje ---")
        print(f"Paquete: {nombre_final}")
        print(f"Fechas Globales: {f_ini_req} al {f_fin_req}")
        print(f"Total a Pagar: ${costo_total}")

        confirmar = input("Confirmar y Reservar? (S/N): ").upper()
        if confirmar == 'S':
            pid = self.paquete_dao.crear_y_retornar_id(nombre_final, f_ini_req, f_fin_req, costo_total)
            if pid:
                if self.reserva_dao.crear(rut_cliente, pid):
                    print(f"Su paquete #{pid} ha sido creado y reservado automaticamente.")
                else:
                    print("Paquete creado, pero fallo la reserva automatica.")
            else:
                print("Error critico al guardar el paquete.")

def print_header(t):
    print("\n" + "="*40)
    print(f" {t}")
    print("="*40)

def main():
    service = AgenciaService()
    sesion_actual = None

    while True:
        if not sesion_actual:
            print_header("VIAJES AVENTURA (LOGIN)")
            print("1. Iniciar Sesion (RUT)")
            print("2. Registrarse (Nuevo Cliente)")
            print("0. Salir")
            op = input("Opcion: ")

            if op == "0": break

            elif op == "1":
                rut = input("RUT: ")
                pw = getpass.getpass("Contrasena: ")
                user = service.usuario_dao.login(rut, pw)
                if user:
                    print(f"\nBienvenido, usuario RUT: {user['rut']}")
                    sesion_actual = user
                else:
                    print("\nCredenciales incorrectas.")

            elif op == "2":
                print("\nRegistro de Nuevo Usuario")
                rut = input("Ingrese su RUT: ")
                pw = getpass.getpass("Cree una contrasena: ")
                if service.usuario_dao.crear(rut, pw, rol_codigo='CUS'):
                    print("Usuario registrado como CLIENTE. Ahora inicie sesion.")
                else:
                    print("Error al registrar. Quizas el RUT ya existe.")

        else:
            rol_txt = sesion_actual['rol'].upper()
            print_header(f"MENU PRINCIPAL ({rol_txt}) - {sesion_actual['rut']}")

            if sesion_actual['rol'] == 'cliente':
                print("1. Armar mi propio viaje (Buscar por Fecha)")
                print("2. Ver Mis Reservas")

            if sesion_actual['rol'] == 'admin':
                print("--- ADMIN ---")
                print("1. Gestionar Destinos (Crear/Editar/Borrar)")
                print("2. Ver Destinos Creados (Inventario)")

            print("0. Cerrar Sesion")
            op = input("Opcion: ")

            if op == "0":
                sesion_actual = None
                print("Sesion cerrada.")

            elif sesion_actual['rol'] == 'cliente':
                if op == "1":
                    service.menu_cliente_armar_viaje(sesion_actual['rut'])
                elif op == "2":
                    reservas = service.reserva_dao.listar_por_usuario(sesion_actual['rut'])
                    print("\n--- MIS RESERVAS ---")
                    if not reservas: print("No tienes reservas.")
                    for r in reservas:
                        print(f"#{r['id']} | {r['paquete']} | Creada: {r['fecha']}")

            elif sesion_actual['rol'] == 'admin':
                if op == "1":
                    service.menu_admin_crud_destinos()
                elif op == "2":
                    destinos = service.destino_dao.listar()
                    print("\n--- INVENTARIO DE DESTINOS ---")
                    for d in destinos:
                         print(f"#{d['id']} | {d['nombre']} | ${d['costo']} | {d['f_ini']} - {d['f_fin']}")

if __name__ == "__main__":
    main()


 VIAJES AVENTURA (LOGIN)
1. Iniciar Sesion (RUT)
2. Registrarse (Nuevo Cliente)
0. Salir
Opcion: 0
